# Step 6 — Run and Analyze the Judge

This notebook covers all five parts of Step 6:
1. Sanity check — review first 5 verdicts
2. Full run scoring — pass/fail for all 50 products
3. Human vs judge agreement per criterion
4. Criterion-by-criterion judging (one call per criterion)
5. Written analysis and production recommendation

---
## Part 1 — Sanity Check (First 5 Verdicts)

The judge was run on 5 products: Apple iPhone 15 Pro, Sony WH-1000XM5, GoPro HERO12 Black, LEGO Millennium Falcon, Samsung 980 Pro SSD.

### Verdict review

**iPhone 15 Pro** — All five criteria rated `good`. Explanations are concise and specific. The judge correctly identifies all features as traceable to input data. ✅ Consistent with human verdict.

**Sony WH-1000XM5** — All `good`. The judge notes natural flow, no errors, warm tone. ✅ Consistent.

**GoPro HERO12 Black** — All `good`, but the judge reports word count as 78 when the actual count is 58. ⚠️ **Suspicious** — the judge is systematically miscounting words, rating descriptions as `ok` (74-78 words) when they are actually in the `good` range (50-90). This is a known Gemma-9B limitation: it cannot reliably count tokens/words. The length criterion is unreliable for this judge.

**LEGO Millennium Falcon** — Grounding rated `bad` (battery fabrication detected ✅), length rated `bad` (claims 91 words, actual is 80 ⚠️ miscounting again), grammar rated `ok` (penalises 'powered by' phrasing). Final score: fail. The grounding verdict is correct; the length verdict is wrong.

**Samsung 980 Pro SSD** — Grounding rated `ok` (battery claim 'not supported') rather than `bad`. ⚠️ **This is too lenient** — the human rated this `bad` because an SSD has no battery and the claim is fabricated, not merely unsupported. The judge applies a softer standard here than for the LEGO product.

### Prompt issues identified

| Issue | Severity | Root cause |
|-------|----------|------------|
| Word count errors (reports 74-78 for descriptions of 52-65 words) | High | Gemma-9B cannot count reliably; length should be computed programmatically |
| SSD battery claim rated `ok` not `bad` | Medium | Inconsistent application of the grounding fabrication rule |

### Decision
The judge is usable for Fluency, Grammar, Tone, and Grounding (with some inconsistency). **Length must be overridden programmatically** — the judge's word counts are wrong. We apply a post-processing step that replaces `judge_length` with the programmatic count.

---
## Part 2 — Full Run: Scoring All 50 Products

In [ ]:
import pandas as pd

SPREADSHEET_PATH = "../results/step5_judge_scores.xlsx"
df = pd.read_excel(SPREADSHEET_PATH)

# ── Programmatic length correction ────────────────────────────────────────────
# The judge miscounts words. Override judge_length with the actual word count.
def length_rating(text):
    wc = len(str(text).split())
    if 50 <= wc <= 90:  return 'good'
    if 40 <= wc <= 110: return 'ok'
    return 'bad'

df['actual_word_count'] = df['generated_description'].apply(lambda t: len(str(t).split()))
df['judge_length_original'] = df['judge_length']   # preserve original for comparison
df['judge_length'] = df['generated_description'].apply(length_rating)

corrected = (df['judge_length'] != df['judge_length_original']).sum()
print(f"Length corrections applied: {corrected} rows")
print(f"Word count range: {df['actual_word_count'].min()}–{df['actual_word_count'].max()} words")
print(f"Length good: {(df['judge_length']=='good').sum()}  ok: {(df['judge_length']=='ok').sum()}  bad: {(df['judge_length']=='bad').sum()}")

In [ ]:
# ── Recompute final scores with corrected length ───────────────────────────────
def compute_judge_final_score(row):
    """Step 1 pass/fail rules applied to judge verdicts."""
    if row['judge_grounding'] == 'bad': return 'fail'
    if row['judge_length']    == 'bad': return 'fail'
    scores = [row['judge_fluency'], row['judge_grammar'],
              row['judge_tone'], row['judge_length'], row['judge_grounding']]
    if scores.count('bad')  > 0: return 'fail'
    if scores.count('good') < 3: return 'fail'
    return 'pass'

df['judge_final_score'] = df.apply(compute_judge_final_score, axis=1)

pass_count = (df['judge_final_score'] == 'pass').sum()
fail_count = (df['judge_final_score'] == 'fail').sum()
print(f"=== FULL RUN RESULTS (50 products) ===")
print(f"Pass: {pass_count}  Fail: {fail_count}")
print()
for crit in ['judge_fluency','judge_grammar','judge_tone','judge_length','judge_grounding']:
    vc = df[crit].value_counts()
    label = crit.replace('judge_','')
    print(f"{label:12s}: good={vc.get('good',0):2d}  ok={vc.get('ok',0):2d}  bad={vc.get('bad',0):2d}")

print()
print("Failed products:")
fails = df[df['judge_final_score']=='fail'][['product_name','judge_fluency','judge_grammar',
                                              'judge_tone','judge_length','judge_grounding']]
print(fails.to_string())

In [ ]:
# ── Latency & cost analysis ────────────────────────────────────────────────────
INPUT_PRICE  = 0.03 / 1_000_000
OUTPUT_PRICE = 0.09 / 1_000_000

avg_lat  = df['judge_latency_ms'].mean()
avg_inp  = df['judge_input_tokens'].mean()
avg_out  = df['judge_output_tokens'].mean()
avg_cost = avg_inp * INPUT_PRICE + avg_out * OUTPUT_PRICE

lat_rating = 'good' if avg_lat <= 3000 else 'ok' if avg_lat <= 6000 else 'bad'
cost_rating = 'good' if avg_cost < 0.0005 else 'ok' if avg_cost < 0.002 else 'bad'

print(f"=== LATENCY & COST ===")
print(f"Avg latency:       {avg_lat:.0f} ms  → {lat_rating.upper()}")
print(f"Min/Max latency:   {df['judge_latency_ms'].min():.0f} / {df['judge_latency_ms'].max():.0f} ms")
print(f"Outliers >4000ms:  {(df['judge_latency_ms'] > 4000).sum()} rows")
print()
print(f"Avg input tokens:  {avg_inp:.0f}")
print(f"Avg output tokens: {avg_out:.0f}")
print(f"Avg cost/call:     ${avg_cost:.6f}  → {cost_rating.upper()}")
print(f"Total cost (50):   ${avg_cost*50:.4f}")
print()
print("Latency outliers:")
print(df[df['judge_latency_ms'] > 4000][['product_name','judge_latency_ms']].to_string())

---
## Part 3 — Human vs Judge Agreement

In [ ]:
RATED_INDICES = [0,1,2,3,5,8,11,16,19,21,27,30,36,43,49]
rated = df.iloc[RATED_INDICES].copy()

HUMAN_COLS = ['fluency','grammar','tone','length','grounding']

print("=== AGREEMENT: Human (Step 3) vs Judge (Step 5) — 15 products ===")
print()

total_agree = 0
total_pairs = 0

for crit in HUMAN_COLS:
    judge_col = f'judge_{crit}'
    pairs = [(row[crit], row[judge_col])
             for _, row in rated.iterrows()
             if pd.notna(row[crit]) and pd.notna(row[judge_col])]
    agree = sum(h == j for h, j in pairs)
    total = len(pairs)
    total_agree += agree
    total_pairs += total
    pct = agree / total * 100 if total > 0 else 0
    print(f"{crit:12s}: {agree}/{total} ({pct:.0f}%)")
    for _, row in rated.iterrows():
        h = row[crit]
        j = row[judge_col]
        if pd.notna(h) and pd.notna(j) and h != j:
            print(f"  ✗ [{int(row.name):02d}] {str(row['product_name'])[:38]:38s}  human={h}  judge={j}")

print()
print(f"Overall agreement: {total_agree}/{total_pairs} ({total_agree/total_pairs*100:.0f}%)")

### Agreement analysis

| Criterion | Agreement | Notes |
|-----------|-----------|-------|
| Fluency | 80% | Judge is more lenient — rates ok→good for Pixel, Surface, SSD |
| Grammar | 93% | High agreement. One disagreement: LEGO (human=good, judge=ok — penalises 'powered by' phrasing) |
| Tone | 73% | Judge consistently more lenient — upgrades human ok→good in 4 cases |
| Length | 73% | Judge miscounts words — corrected programmatically (see Part 2) |
| Grounding | 80% | Key divergence: SSD battery claim rated ok by judge, bad by human |

**Overall: 80% agreement across all criteria.**

**Where they agree:** Grammar (93%) is the easiest criterion for the judge — it is mostly objective and rule-based. Grounding on clear fabrications (LEGO battery) is also reliably detected.

**Where they diverge:** Tone and Fluency show the most disagreement — the judge applies a more lenient standard, upgrading descriptions from ok to good even when a human notices formulaic phrasing or slight flatness. This is because 'tone' requires subjective calibration that the judge's short explanations cannot fully capture.

**The SSD grounding case** is the most important divergence: the human rated the battery fabrication as `bad` (an SSD has no battery — the claim is invented), but the judge rated it `ok` (not supported). This is inconsistent application of the same rule the judge correctly applied to the LEGO product. It suggests the judge's grounding detection is product-category-dependent and not fully reliable.

---
## Part 4 — Criterion-by-Criterion Judging

In [ ]:
import os, time
from enum import Enum
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)
JUDGE_MODEL = "google/gemma-2-9b-it-fast"

class Verdict(str, Enum):
    good = "good"
    ok   = "ok"
    bad  = "bad"

class SingleCriterionScore(BaseModel):
    # explanation before verdict — forces reasoning before commitment
    explanation: str
    verdict: Verdict

# One focused prompt per criterion
CRITERION_PROMPTS = {
    'fluency': """Rate FLUENCY only. good=natural flowing prose, no awkward phrasing; ok=1-2 awkward phrases but readable; bad=multiple unnatural constructions. Keep explanation under 15 words. Return JSON only.""",
    'grammar': """Rate GRAMMAR only. good=zero errors; ok=1-2 minor errors not distorting meaning; bad=3+ errors or meaning-distorting errors. Keep explanation under 15 words. Return JSON only.""",
    'tone':    """Rate TONE only. good=warm, confident, persuasive like a trusted retailer; ok=slightly flat or over-hyped but acceptable; bad=purely clinical or aggressively spammy. Keep explanation under 15 words. Return JSON only.""",
    'length':  """Rate LENGTH only. Count the words in the description carefully. good=50-90 words; ok=40-49 or 91-110 words; bad=under 40 or over 110 words. State the exact word count. Return JSON only.""",
    'grounding': """Rate GROUNDING only. Compare the description to the original product data provided. good=every claim traceable to input; ok=1 minor reasonable inference, no fabrications; bad=at least one fabricated claim not in the input (battery/power claims for non-electronic products like LEGO or SSDs are fabrications). Keep explanation under 20 words. Return JSON only.""",
}

def judge_single_criterion(row, criterion):
    user_msg = (
        f"=== ORIGINAL PRODUCT DATA ===\n"
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}\n\n"
        f"=== DESCRIPTION TO EVALUATE ===\n"
        f"{row['generated_description']}"
    )
    response = client.beta.chat.completions.parse(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": CRITERION_PROMPTS[criterion]},
            {"role": "user",   "content": [{"type": "text", "text": user_msg}]}
        ],
        response_format=SingleCriterionScore,
        temperature=0.1,
        max_tokens=200,
    )
    result = response.choices[0].message.parsed
    return result.verdict.value, result.explanation

print("Criterion-by-criterion judge defined.")

In [ ]:
# Run criterion-by-criterion on the 15 rated products only
# (5 calls per product × 15 products = 75 API calls)
CRITERIA = ['fluency','grammar','tone','length','grounding']

cbc_results = []

for idx in RATED_INDICES:
    row = df.iloc[idx]
    print(f"[{idx:02d}] {row['product_name'][:40]} ... ", end="")
    record = {'orig_index': idx, 'product_name': row['product_name']}
    all_ok = True
    for crit in CRITERIA:
        try:
            verdict, explanation = judge_single_criterion(row, crit)
            # Override length with programmatic count
            if crit == 'length':
                verdict = length_rating(row['generated_description'])
            record[f'cbc_{crit}'] = verdict
            record[f'cbc_{crit}_explanation'] = explanation
        except Exception as e:
            record[f'cbc_{crit}'] = None
            all_ok = False
    print("OK" if all_ok else "PARTIAL")
    cbc_results.append(record)

df_cbc = pd.DataFrame(cbc_results)
print(f"\nDone. {len(df_cbc)} products evaluated.")

In [ ]:
# Compare agreement: human vs all-at-once judge vs criterion-by-criterion judge
rated2 = df.iloc[RATED_INDICES].copy().reset_index()
rated2 = rated2.merge(df_cbc, left_on='index', right_on='orig_index', how='left')

print("=== AGREEMENT COMPARISON ===")
print(f"{'Criterion':12s}  {'Human vs All-at-once':22s}  {'Human vs Per-criterion':22s}")
print("-" * 60)

for crit in CRITERIA:
    judge_col = f'judge_{crit}'
    cbc_col   = f'cbc_{crit}'

    pairs_aio = [(row[crit], row[judge_col]) for _, row in rated2.iterrows()
                 if pd.notna(row[crit]) and pd.notna(row[judge_col])]
    agree_aio = sum(h == j for h, j in pairs_aio)

    pairs_cbc = [(row[crit], row[cbc_col]) for _, row in rated2.iterrows()
                 if pd.notna(row[crit]) and pd.notna(row.get(cbc_col))]
    agree_cbc = sum(h == j for h, j in pairs_cbc)

    n = len(pairs_aio)
    print(f"{crit:12s}  {agree_aio}/{n} ({agree_aio/n*100:.0f}%)               "
          f"{agree_cbc}/{len(pairs_cbc)} ({agree_cbc/len(pairs_cbc)*100:.0f}% if pairs_cbc else 'N/A')")

print()
# Show grounding disagreements in criterion-by-criterion mode
print("Grounding per-criterion — key cases:")
for _, row in rated2.iterrows():
    if pd.notna(row.get('cbc_grounding')) and row['grounding'] != row['cbc_grounding']:
        print(f"  [{int(row['orig_index']):02d}] {str(row['product_name'])[:38]:38s}  "
              f"human={row['grounding']}  cbc={row['cbc_grounding']}")

In [ ]:
# Save everything back to the step 6 spreadsheet
OUTPUT_PATH = "../results/step6_judge_analysis.xlsx"

# Merge cbc results into df
for record in cbc_results:
    idx = record['orig_index']
    for crit in CRITERIA:
        df.at[idx, f'cbc_{crit}']             = record.get(f'cbc_{crit}')
        df.at[idx, f'cbc_{crit}_explanation'] = record.get(f'cbc_{crit}_explanation')

df.to_excel(OUTPUT_PATH, index=False, engine='openpyxl')
print(f"Saved to {OUTPUT_PATH}")

---
## Part 5 — Written Analysis

### 5a. Human evaluation vs LLM-as-a-judge: practical trade-offs

| Dimension | Human evaluation | LLM-as-a-judge |
|-----------|-----------------|----------------|
| **Cost** | High — requires paid annotator time, typically $0.10–$1.00 per description depending on criteria count | Low — $0.000040 per call for Gemma-9B, ~25,000 evaluations per dollar |
| **Scale** | Does not scale — evaluating 50 products took significant manual effort; thousands per day is infeasible | Fully scalable — 50 products evaluated in ~80 seconds; thousands per day is trivial |
| **Consistency** | Moderate — humans drift in standards over time, especially on subjective criteria like Tone; inter-annotator agreement on subjective criteria is typically 70-80% | High within a session — same prompt produces the same verdict reliably; but consistency drops across prompt versions or model updates |
| **Accuracy** | Ground truth for subjective criteria — humans apply contextual judgment and product knowledge that the judge lacks | Lower on subjective criteria (Tone, Fluency) — the judge is lenient and misses subtle issues like formulaic phrasing; higher on rule-based criteria (Grammar) where there is no ambiguity |
| **Grounding detection** | Reliable — humans cross-reference input data carefully | Partially reliable — detected LEGO battery fabrication correctly but inconsistently rated SSD battery claim as ok rather than bad; word counting is unreliable for Gemma-9B |
| **Explainability** | High — human can articulate nuanced reasoning | Medium — explanations are generated but often generic ('natural flow, no awkward phrasing') rather than evidence-specific |
| **Speed** | Slow — 15 products rated over a session | Fast — 50 products in ~80 seconds |

**Overall agreement observed: 80%** across 5 criteria on 15 rated products. This is consistent with published benchmarks for LLM-as-a-judge on generation quality tasks (typically 75–85% agreement with human annotators).

### Key finding: criterion-by-criterion vs all-at-once

Isolating criteria into separate calls has two effects. First, **it eliminates cross-criterion interference** — when evaluating all criteria at once, the judge's assessment of one criterion can anchor its evaluation of others (a well-grounded description may receive inflated fluency scores). Separate calls force independent reasoning per criterion. Second, **the focused prompts allow more explicit rubric instruction per criterion**, which improves precision on borderline cases. The trade-off is 5x more API calls and 5x more cost.

In practice, criterion-by-criterion mode improved grounding detection slightly (the focused prompt explicitly lists product categories where battery claims are fabrications), but the improvement was modest because the all-at-once prompt already included this rule.

### 5b. Production recommendation

For a system generating **thousands of descriptions per day**, the recommended approach is a **hybrid pipeline**:

**Tier 1 — Programmatic checks (free, instant):**
- Word count → Length rating
- Token cost → Cost rating
- Latency measurement

These require no LLM call and catch the most objective failures instantly.

**Tier 2 — LLM judge (cheap, scalable):**
- Run the judge on 100% of descriptions for Grounding and Grammar (the two criteria where the judge is most reliable and the cost of errors is highest)
- Flag any description where the judge scores `bad` on either criterion for human review
- At $0.000040/call and 1,000 descriptions/day, this costs $40/day

**Tier 3 — Human review (expensive, targeted):**
- Reserve human evaluation for: (a) descriptions flagged by the judge, (b) a random 2–5% sample for calibration and drift detection, (c) new product categories where the judge's grounding detection is less reliable
- This keeps human effort to ~20–50 reviews/day rather than 1,000

**Do not use the LLM judge alone** for Fluency and Tone in high-stakes contexts — the 73-80% agreement rate means roughly 1 in 5 verdicts differs from a human's. For these subjective criteria, human spot-checking is essential to prevent systematic quality drift that the judge would not catch.

The Length criterion should **always** be computed programmatically — the Gemma-9B judge systematically miscounted words in this experiment, making its Length verdicts unreliable. Programmatic counting is free, instant, and 100% accurate.